# Demo 03 — Why Models Fail

**Class 01 · D5-D9** (slides: `slides/01_foundations.md`)

* **D5** Under- and overfitting and the bias-variance trade-off (Example 1 returns).
* **D6** No Free Lunch: no model wins everywhere.
* **D7** Data leakage: accuracy on pure noise.
* **D8** Correlation is not causation.
* **D9** Spurious correlations (shortcuts) under distribution shift.

D5 uses Example 1, a noisy sine with $n = 50$ and $\sigma = 0.3$. D6-D9 use their own toy data, labelled as such.
Every cell that prints a number of a slide starts with `slide: <frame title>`; plots go to `assets/figures/plots/01/`.
(The curse of dimensionality moves to Class 06.)

In [ ]:
import numpy as np
import polars as pl
from numpy.polynomial import Polynomial
from sklearn.datasets import make_circles, make_classification, make_moons
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

from faa import plotlib

plotlib.setup(module="01")
BLUE, ORANGE, GREEN, RED, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.PURPLE,
    plotlib.GREY,
)
SEED, SIGMA = 42, 0.3
pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(20)
pl.Config.set_tbl_width_chars(160)


def f(x):
    """The true pattern of Example 1."""
    return np.sin(2 * np.pi * x)


def example1(n=50, sigma=SIGMA, seed=SEED, n_test=1000):
    """Example 1: a noisy sine. Fresh generator per call, so any section can be rerun alone."""
    rng = np.random.default_rng(seed)
    x = rng.uniform(0, 1, n)
    y = f(x) + rng.normal(0, sigma, n)
    x_test = rng.uniform(0, 1, n_test)
    y_test = f(x_test) + rng.normal(0, sigma, n_test)
    return x, y, x_test, y_test


def mse(y, y_hat):
    return float(np.mean((y - y_hat) ** 2))


def show(table: pl.DataFrame, digits: int = 4) -> None:
    """Print a table with rounded floats."""
    print(table.with_columns(pl.col(pl.Float64).round(digits)))


def slide(title: str) -> None:
    print(f"--- slide: {title} ---")


grid = np.linspace(0, 1, 400)

## D5 — Underfitting, overfitting and the bias-variance trade-off

### Three fits of Example 1 (slide: Underfitting, a Good Fit, Overfitting; plot `d5_fits`)

In [ ]:
# slide: Underfitting, a Good Fit, Overfitting
slide("Underfitting, a Good Fit, Overfitting")
x_train, y_train, x_test, y_test = example1()
fits = {}
rows = []
for q, label in [(1, "underfitting"), (3, "a good fit"), (15, "overfitting")]:
    p = Polynomial.fit(x_train, y_train, deg=q)
    fits[q] = (p, label)
    rows.append(
        {"degree q": q, "fit": label, "train_mse": mse(y_train, p(x_train)), "test_mse": mse(y_test, p(x_test))}
    )
show(pl.DataFrame(rows))
print(f"noise floor sigma^2 = {SIGMA**2:.2f}")

In [ ]:
# plot d5_fits
fg, axs = plotlib.fig("full", 1.8, ncols=3, sharey=True)
for ax, (q, (p, label)), row in zip(axs, fits.items(), rows):
    ax.plot(grid, f(grid), color=GREEN, ls="--", lw=1)
    ax.scatter(x_train, y_train, s=5, color=GREY, zorder=2)
    ax.plot(grid, p(grid), color=BLUE if q == 3 else RED, lw=1.5, zorder=3)
    ax.set_title(f"$q$ = {q}: {label}\ntrain {row['train_mse']:.3f}, test {row['test_mse']:.3f}", fontsize=7.5)
    ax.set(xlabel="$x$", ylim=(-1.8, 1.8), xlim=(0, 1))
axs[0].set_ylabel("$y$")
plotlib.save(fg, "d5_fits")

### Error against model complexity (slide: Error Versus Model Complexity; plot `d5_error_curve`)

In [ ]:
# slide: Error Versus Model Complexity
slide("Error Versus Model Complexity")
rows = []
for q in range(16):
    p = Polynomial.fit(x_train, y_train, deg=q)
    rows.append({"q": q, "train_mse": mse(y_train, p(x_train)), "test_mse": mse(y_test, p(x_test))})
curve = pl.DataFrame(rows)
show(curve)
q_best = int(np.argmin(curve["test_mse"].to_numpy()))
print(
    f"lowest test error at q = {q_best} ({curve['test_mse'][q_best]:.4f}); the training error keeps falling to {curve['train_mse'][15]:.4f} at q = 15"
)

In [ ]:
# plot d5_error_curve
fg, ax = plotlib.fig("full", 1.7)
ax.plot(curve["q"], curve["train_mse"], color=BLUE, marker="o", ms=2.5, label="training error")
ax.plot(curve["q"], curve["test_mse"], color=ORANGE, marker="s", ms=2.5, label="test error")
ax.axhline(SIGMA**2, color=GREY, ls=":", lw=1)
ax.annotate(
    "noise floor", (0, SIGMA**2), (2, -3), textcoords="offset points", ha="left", va="top", fontsize=7.5, color=GREY
)
ax.axvline(q_best, color=RED, ls="--", lw=1)
ax.annotate(
    f"lowest test error, $q$ = {q_best}", (q_best, 2.2), (4, 0), textcoords="offset points", fontsize=7.5, color=RED
)
ax.set(yscale="log", xlabel="polynomial degree $q$", ylabel="MSE (log scale)", xlim=(-0.3, 15.3))
ax.legend(
    loc="lower center", bbox_to_anchor=(0.5, 1.0), ncols=2, borderaxespad=0.1, columnspacing=1.0, handletextpad=0.3
)
plotlib.save(fg, "d5_error_curve")

### Bias and variance: 200 training sets from the same process

Each training set is a fresh draw of Example 1 (new positions $x$ and new noise), the same model is fitted to each set, error measured on a grid.
$\text{bias}^2$ = squared distance of the **average** fit from the truth, variance = how much the fits **disagree**, and
$\text{expected error} = \text{bias}^2 + \text{variance} + \sigma^2$ (the simulated test error below confirms it).
Degrees 0 to 10: above that the variance of a degree-14 fit on 50 random points is dominated by a few wild fits and the average is not stable.

In [ ]:
# slide: The Bias-Variance Decomposition; The Trade-Off and What to Do About It
slide("Bias and Variance; The Bias-Variance Decomposition")


def bias_variance(n=50, sigma=SIGMA, degrees=range(11), n_sets=200, seed=SEED):
    """Bias^2, variance and simulated test error per degree: a fresh Example 1 training set for each fit."""
    rng = np.random.default_rng(seed)
    xs = rng.uniform(0, 1, size=(n_sets, n))
    g = np.linspace(0, 1, 200)
    truth = f(g)
    ys = f(xs) + rng.normal(0, sigma, size=(n_sets, n))
    y_new = truth + rng.normal(0, sigma, size=(n_sets, len(g)))  # a fresh noisy target for each fit
    rows, curves = [], {}
    for q in degrees:
        q_fits = np.array([Polynomial.fit(x_i, y_i, deg=q)(g) for x_i, y_i in zip(xs, ys)])
        b2 = float(np.mean((q_fits.mean(axis=0) - truth) ** 2))
        var = float(np.mean(q_fits.var(axis=0)))
        rows.append({"q": q, "bias2": b2, "variance": var, "noise": sigma**2, "sum": b2 + var + sigma**2,
                     "test_mse_simulated": float(np.mean((y_new - q_fits) ** 2))})  # fmt: skip
        curves[q] = q_fits
    return pl.DataFrame(rows), curves, g


bv, bv_fits, bv_grid = bias_variance()
show(bv, 4)
q_star = int(np.argmin(bv["sum"].to_numpy()))
print(f"the degree with the lowest bias^2 + variance + noise: q = {q_star}")
sel = bv.filter(pl.col("q").is_in([1, 3, 9]))
show(sel, 4)

In [ ]:
# plot d5_bias_variance: 30 of the 200 fits for q = 1 and q = 9
fg, axs = plotlib.fig("full", 1.8, ncols=2, sharey=True)
for ax, q, name in [(axs[0], 1, "high bias"), (axs[1], 9, "high variance")]:
    for fit in bv_fits[q][:30]:
        ax.plot(bv_grid, fit, color=BLUE, alpha=0.18, lw=0.8)
    ax.plot(bv_grid, bv_fits[q].mean(axis=0), color=RED, lw=1.6, label="average fit")
    ax.plot(bv_grid, f(bv_grid), color="k", ls="--", lw=1.2, label="truth")
    row = bv.row(q, named=True)
    ax.set_title(f"$q$ = {q}: {name}\nbias$^2$ {row['bias2']:.3f}, variance {row['variance']:.3f}", fontsize=7.5)
    ax.set(xlabel="$x$", ylim=(-2, 2), xlim=(0, 1))
axs[0].set_ylabel("$y$")
axs[0].legend(loc="upper right", borderaxespad=0.1, handletextpad=0.3)
plotlib.save(fg, "d5_bias_variance")

In [ ]:
# plot d5_tradeoff
fg, ax = plotlib.fig("full", 1.8)
ax.plot(bv["q"], bv["bias2"], color=BLUE, marker="o", ms=2.5, label="bias$^2$")
ax.plot(bv["q"], bv["variance"], color=ORANGE, marker="s", ms=2.5, label="variance")
ax.axhline(SIGMA**2, color=GREY, ls=":", lw=1, label="noise $\\sigma^2$")
ax.plot(bv["q"], bv["sum"], color=PURPLE, lw=1.8, marker="D", ms=3, label="bias$^2$ + variance + noise")
ax.axvline(q_star, color=RED, ls="--", lw=1)
ax.set(yscale="log", xlabel="polynomial degree $q$", ylabel="error (log scale)", xlim=(-0.3, 10.3))
ax.legend(
    loc="lower center",
    bbox_to_anchor=(0.5, 1.0),
    ncols=4,
    borderaxespad=0.1,
    columnspacing=0.6,
    handletextpad=0.3,
    fontsize=7.5,
)
plotlib.save(fg, "d5_tradeoff")

### Sweep: sample size and noise (no slide)

100 training sets per cell. Columns: the degree that minimizes bias$^2$ + variance + noise, and bias$^2$ and variance of $q = 1$, $q = 3$ and $q = 9$.
More data lowers the variance, more noise raises it: the best degree depends on both.

In [ ]:
rows = []
for n_s in (10, 20, 50, 100):
    for sigma_s in (0.1, 0.3, 1.0):
        t, _, _ = bias_variance(n_s, sigma_s, degrees=range(min(10, n_s)), n_sets=100)
        row = {"n": n_s, "sigma": sigma_s, "best_q": int(np.argmin(t["sum"].to_numpy()))}
        for q in (1, 3, 6):
            r = t.row(q, named=True)
            row[f"bias2_q{q}"], row[f"var_q{q}"] = r["bias2"], r["variance"]
        rows.append(row)
show(pl.DataFrame(rows), 4)

## D6 — No Free Lunch

Averaged over *all* possible problems, every learning algorithm performs the same. A model is good only when its **assumptions match the problem**.
Three toy datasets (labelled as such: they are not Example 1 or Example 2) and three models, 5-fold CV repeated 4 times (20 scores per cell).

### The ranking flips (slide: No Free Lunch and Inductive Bias; plot `d6_ranking`)

In [ ]:
# slide: No Free Lunch and Inductive Bias
slide("No Free Lunch and Inductive Bias")
datasets = {
    "concentric circles": make_circles(n_samples=400, noise=0.1, factor=0.4, random_state=SEED),
    "two moons": make_moons(n_samples=400, noise=0.3, random_state=SEED),
    "linear, 10 of 50 useful": make_classification(
        n_samples=200, n_features=50, n_informative=10, n_redundant=0, n_clusters_per_class=1, random_state=SEED
    ),
}
models = {
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "k-NN (k = 5)": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "decision tree": DecisionTreeClassifier(random_state=SEED),
}
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=SEED)
rows = []
for dname, (Xd, yd) in datasets.items():
    for mname, model in models.items():
        sc = cross_val_score(model, Xd, yd, cv=rcv)
        rows.append({"dataset": dname, "model": mname, "accuracy": sc.mean(), "sd": sc.std(ddof=1)})
rank = pl.DataFrame(rows)
show(rank, 3)
for dname in datasets:
    sub = rank.filter(pl.col("dataset") == dname).sort("accuracy", descending=True)
    print(
        f"{dname:26s} best: {sub['model'][0]} ({sub['accuracy'][0]:.3f}), worst: {sub['model'][-1]} ({sub['accuracy'][-1]:.3f})"
    )

In [ ]:
# plot d6_ranking
fg, ax = plotlib.fig("full", 1.9)
names = list(datasets)
xs, w = np.arange(len(names)), 0.26
for j, (mname, colour) in enumerate(zip(models, (BLUE, ORANGE, GREEN))):
    sub = rank.filter(pl.col("model") == mname)
    vals = [sub.filter(pl.col("dataset") == d)["accuracy"][0] for d in names]
    bars = ax.bar(xs + (j - 1) * w, vals, w, color=colour, label=mname)
    for rect, v in zip(bars, vals):
        ax.annotate(f"{v:.2f}", (rect.get_x() + w / 2, v), (0, 2), textcoords="offset points", ha="center", fontsize=7)
ax.axhline(0.5, color=GREY, ls=":", lw=1)
ax.set(xticks=xs, xticklabels=names, ylim=(0.4, 1.12), ylabel="CV accuracy")
ax.legend(
    loc="lower center", bbox_to_anchor=(0.5, 1.0), ncols=3, borderaxespad=0.1, columnspacing=1.0, handletextpad=0.3
)
ax.grid(axis="x", visible=False)
plotlib.save(fg, "d6_ranking")

The ranking **flips** between problems: the straight boundary of logistic regression is the right assumption for the last dataset and the wrong one for the circles.
There is no universally best model, only models whose inductive bias fits the data.

## D7 — Data leakage: accuracy on pure noise

100 samples, 10 000 features and **random** labels. There is nothing to learn: the honest accuracy is 50%.
A common mistake is to select the "best" 20 features using **all** the data and only afterwards run cross-validation.

### Wrong and right pipelines (slides: What Is Data Leakage?, Wrong and Right Pipelines; plot `d7_leak`)

In [ ]:
# slide: What Is Data Leakage?; Wrong and Right Pipelines
slide("Wrong and Right Pipelines")
rng = np.random.default_rng(SEED)
X_noise = rng.normal(size=(100, 10_000))
y_noise = rng.integers(0, 2, size=100)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
clf = LogisticRegression(max_iter=1000)
# Wrong: feature selection sees the test folds
X_selected = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)
leaky = cross_val_score(clf, X_selected, y_noise, cv=cv).mean()
# Right: feature selection is part of the model, refit inside each training fold
honest = cross_val_score(make_pipeline(SelectKBest(f_classif, k=20), clf), X_noise, y_noise, cv=cv).mean()
print(f"labels are random: {int(y_noise.sum())} ones and {int((1 - y_noise).sum())} zeros")
print(f"feature selection BEFORE cross-validation: accuracy = {leaky:.3f}   <- leakage")
print(f"feature selection INSIDE cross-validation: accuracy = {honest:.3f}   <- honest")
chance = max(y_noise.mean(), 1 - y_noise.mean())
print(f"majority-class accuracy (chance) = {chance:.2f}")

### Sweep: how many features, how many samples? (no slide)

Fresh noise for every $n$ (3 repetitions), selection of the $k$ best features by the F statistic. "Leaky" selects on all the data, "honest" inside each training fold.
The leak shrinks when $n$ grows, and grows when more features are kept.

In [ ]:
rng = np.random.default_rng(SEED)
ks = (1, 5, 20, 100, 500)
rows = []
for n_s in (50, 100, 200, 500):
    acc = {k: ([], []) for k in ks}
    for rep in range(3):
        Xn = rng.normal(size=(n_s, 10_000))
        yn = rng.integers(0, 2, size=n_s)
        cv_n = StratifiedKFold(5, shuffle=True, random_state=rep)
        order_all = np.argsort(-f_classif(Xn, yn)[0])
        for k in ks:
            acc[k][0].append(
                cross_val_score(LogisticRegression(max_iter=1000), Xn[:, order_all[:k]], yn, cv=cv_n).mean()
            )
        honest_k = {k: [] for k in ks}
        for tr, te in cv_n.split(Xn, yn):
            order = np.argsort(-f_classif(Xn[tr], yn[tr])[0])
            for k in ks:
                honest_k[k].append(
                    LogisticRegression(max_iter=1000)
                    .fit(Xn[tr][:, order[:k]], yn[tr])
                    .score(Xn[te][:, order[:k]], yn[te])
                )
        for k in ks:
            acc[k][1].append(np.mean(honest_k[k]))
    for k in ks:
        rows.append({"n": n_s, "k_selected": k, "leaky": np.mean(acc[k][0]), "honest": np.mean(acc[k][1])})
leak = pl.DataFrame(rows)
show(leak, 3)

In [ ]:
# plot d7_leak: left the two pipelines of the slide, right the leak against n for several numbers of selected features
fg, axs = plotlib.fig("full", 1.8, ncols=2, gridspec_kw={"width_ratios": [0.85, 1.15]})
ax = axs[0]
bars = ax.bar([0, 1], [leaky, honest], 0.55, color=[RED, BLUE])
for rect, v in zip(bars, [leaky, honest]):
    ax.annotate(f"{v:.2f}", (rect.get_x() + 0.275, v), (0, 2), textcoords="offset points", ha="center", fontsize=7.5)
ax.axhline(0.5, color="k", ls=":", lw=1)
ax.set(
    xticks=[0, 1],
    xticklabels=["select, then\nvalidate", "select in\neach fold"],
    ylim=(0, 1.1),
    ylabel="accuracy (dotted: chance)",
    xlim=(-0.5, 1.5),
)
ax.tick_params(axis="x", labelsize=7)
ax.grid(axis="x", visible=False)
ax = axs[1]
for k, colour in zip(ks, (GREY, BLUE, ORANGE, GREEN, PURPLE)):
    sub = leak.filter(pl.col("k_selected") == k)
    ax.plot(sub["n"], sub["leaky"], marker="o", ms=2.5, color=colour)
    ax.annotate(
        f"{k}", (500, sub["leaky"][-1]), (4, 0), textcoords="offset points", va="center", fontsize=7, color=colour
    )
ax.annotate("features kept", (500, 1.0), (4, 9), textcoords="offset points", fontsize=7, ha="left")
ax.axhline(0.5, color="k", ls=":", lw=1)
ax.set(
    xscale="log",
    xlabel="samples $n$ (log scale), 10 000 features",
    ylabel="leaky accuracy",
    xlim=(45, 700),
    ylim=(0.45, 1.05),
)
fg.subplots_adjust(wspace=0.42)
ax.set_xticks([50, 100, 200, 500], ["50", "100", "200", "500"])
ax.minorticks_off()
plotlib.save(fg, "d7_leak")

**Rule:** any step that learns from data (scaling, feature selection, imputation, tuning) is fitted on the training part only.
Other forms of leakage: duplicated samples across splits, features that encode the target (a "treatment given" column), and using the future to predict the past in time series.

## D8 — Correlation is not causation

A beach town records, for 365 days, the **temperature**, the **ice-cream sales** and the number of **drowning incidents** (a toy dataset).
The true mechanism, which the model never sees: hot days sell more ice cream, hot days bring more swimmers and so more drownings, and ice cream has **no effect** on drownings.

### Correlation and intervention (slides: Correlation and the Ice-Cream Example, Prediction Versus Intervention; plots `d8_scatter`, `d8_intervention`)

In [ ]:
# slide: Correlation and the Ice-Cream Example; The Hidden Common Cause; Prediction Versus Intervention
slide("Correlation and the Ice-Cream Example; Prediction Versus Intervention")
DAYS = 365


def drownings_given(temp, rng):
    """Drownings depend on the temperature only."""
    return np.maximum(0, 0.3 * temp - 3 + rng.normal(0, 1, len(temp)))


rng = np.random.default_rng(SEED)
temperature = rng.uniform(10, 35, DAYS)
ice_cream = 20 * temperature + rng.normal(0, 60, DAYS)
drownings = drownings_given(temperature, rng)
r_ice = np.corrcoef(ice_cream, drownings)[0, 1]
model_ice = LinearRegression().fit(ice_cream[:, None], drownings)
print(
    f"correlation r(ice cream, drownings) = {r_ice:.3f}; r(temperature, ice cream) = {np.corrcoef(temperature, ice_cream)[0, 1]:.3f}; r(temperature, drownings) = {np.corrcoef(temperature, drownings)[0, 1]:.3f}"
)
print(
    f"model 'drownings ~ ice cream': R^2 = {model_ice.score(ice_cream[:, None], drownings):.3f}, weight = {model_ice.coef_[0] * 100:+.4f} drownings per 100 sales"
)
before = drownings.sum()
predicted_after = model_ice.predict(ice_cream[:, None] / 2).sum()
actual_after = drownings_given(temperature, rng).sum()
print(f"drownings per year before the intervention   : {before:6.1f}")
print(f"model's prediction with half the ice cream   : {predicted_after:6.1f}")
print(f"what actually happens with half the ice cream: {actual_after:6.1f}")
both = LinearRegression().fit(np.column_stack([ice_cream, temperature]), drownings)
print(
    f"with the confounder: weight of ice cream = {both.coef_[0] * 100:+.4f} per 100 sales (about 0), weight of temperature = {both.coef_[1]:+.4f} per degree"
)
print(
    f"                     R^2 = {both.score(np.column_stack([ice_cream, temperature]), drownings):.3f} (temperature alone: {LinearRegression().fit(temperature[:, None], drownings).score(temperature[:, None], drownings):.3f})"
)

In [ ]:
# plot d8_scatter
fg, axs = plotlib.fig("full", 1.7, ncols=2, sharey=True)
axs[0].scatter(ice_cream, drownings, s=5, color=BLUE, alpha=0.7)
axs[0].set_title(f"strong correlation, $r$ = {r_ice:.2f}", fontsize=7.5)
sc = axs[1].scatter(ice_cream, drownings, c=temperature, cmap="coolwarm", s=5)
axs[1].set_title("...explained by temperature", fontsize=7.5)
cb = fg.colorbar(sc, ax=axs[1], pad=0.02, aspect=14)
cb.set_label("temperature (°C)", fontsize=7.5)
cb.ax.tick_params(labelsize=7)
for ax in axs:
    ax.set(xlabel="ice-cream sales per day")
axs[0].set_ylabel("drownings per day")
plotlib.save(fg, "d8_scatter")

In [ ]:
# plot d8_intervention
fg, ax = plotlib.fig("full", 1.6)
vals = [before, predicted_after, actual_after]
bars = ax.bar(range(3), vals, 0.55, color=[GREY, ORANGE, BLUE])
for rect, v in zip(bars, vals):
    ax.annotate(f"{v:.0f}", (rect.get_x() + 0.275, v), (0, 2), textcoords="offset points", ha="center", fontsize=7.5)
ax.set(
    xticks=range(3),
    xticklabels=["today", "model's prediction:\nhalf the ice cream", "what happens:\nhalf the ice cream"],
    ylabel="drownings per year",
    ylim=(0, max(vals) * 1.15),
)
ax.grid(axis="x", visible=False)
plotlib.save(fg, "d8_intervention")

### Sweep: a noisy measurement of the confounder (no slide)

Adding the temperature removes the effect of ice cream only if the temperature is measured well. With a thermometer error of $\tau$ degrees the weight of ice cream is not 0.

In [ ]:
rng = np.random.default_rng(SEED)
rows = []
for tau in (0, 2, 5, 10, 20):
    t_meas = temperature + rng.normal(0, tau, DAYS)
    m = LinearRegression().fit(np.column_stack([ice_cream, t_meas]), drownings)
    rows.append({"thermometer_error_sd": tau, "ice_weight_per_100_sales": m.coef_[0] * 100, "temp_weight": m.coef_[1],
                 "R2": m.score(np.column_stack([ice_cream, t_meas]), drownings)})  # fmt: skip
show(pl.DataFrame(rows), 4)
print(f"for comparison, ice cream alone: weight = {model_ice.coef_[0] * 100:+.4f} per 100 sales")

A model trained on **observational** data learns associations. It predicts well as long as the world does not change, but it cannot tell what happens when we **act** on a feature.
Once the confounder is included (and measured well), the weight of ice cream drops to about zero.

## D9 — Spurious correlations and distribution shift

A classifier for a toy two-class problem with two features:

* `core`: truly related to the label, but **noisy**;
* `shortcut`: for example the background of a photo or the hospital that took the X-ray. In the **training** data it agrees with the label 95% of the time; in **deployment** that relationship is gone.

### The shift (slides: Distribution Shift and Its Kinds, Sampling Bias; plots `d9_shift`, `d9_accuracy`)

In [ ]:
# slide: Distribution Shift and Its Kinds; Sampling Bias
slide("Distribution Shift and Its Kinds; Sampling Bias")


def make_data(n, shortcut_agreement, rng):
    """Labels in {0, 1}; the shortcut agrees with the label with the given probability."""
    labels = rng.integers(0, 2, size=n)
    core = (2 * labels - 1) + rng.normal(0, 1.5, n)
    agrees = rng.uniform(size=n) < shortcut_agreement
    shortcut = np.where(agrees, 2 * labels - 1, -(2 * labels - 1)) + rng.normal(0, 0.3, n)
    return np.column_stack([core, shortcut]), labels


rng = np.random.default_rng(SEED)
X_tr, y_tr = make_data(2000, 0.95, rng)
X_iid, y_iid = make_data(2000, 0.95, rng)
X_shift, y_shift = make_data(2000, 0.5, rng)
model = LogisticRegression().fit(X_tr, y_tr)
core_only = LogisticRegression().fit(X_tr[:, :1], y_tr)
print(f"weights of the model with both features: core = {model.coef_[0, 0]:.2f}, shortcut = {model.coef_[0, 1]:.2f}")
acc9 = pl.DataFrame(
    {
        "model": ["both features", "core feature only"],
        "same_distribution": [model.score(X_iid, y_iid), core_only.score(X_iid[:, :1], y_iid)],
        "after_shift": [model.score(X_shift, y_shift), core_only.score(X_shift[:, :1], y_shift)],
    }
)
show(acc9, 3)
print(
    f"accuracy lost by the shift: both features {acc9['same_distribution'][0] - acc9['after_shift'][0]:.3f}, core only {acc9['same_distribution'][1] - acc9['after_shift'][1]:.3f}"
)

In [ ]:
# plot d9_shift
fg, axs = plotlib.fig("full", 1.9, ncols=2, sharex=True, sharey=True)
for ax, (Xd, yd, title) in zip(
    axs, [(X_tr, y_tr, "training: shortcut agrees 95%"), (X_shift, y_shift, "deployment: shortcut agrees 50%")]
):
    for label, colour, mk in [(0, BLUE, "o"), (1, ORANGE, "s")]:
        ax.scatter(
            Xd[yd == label, 0], Xd[yd == label, 1], s=4, color=colour, marker=mk, alpha=0.5, label=f"class {label}"
        )
    xs = np.linspace(-6, 6, 10)
    ax.plot(xs, -(model.coef_[0, 0] * xs + model.intercept_[0]) / model.coef_[0, 1], "k--", lw=1.2)
    ax.set_title(title, fontsize=7.5)
    ax.set(xlabel="core feature", ylim=(-2.5, 2.5), xlim=(-6, 6))
axs[0].set_ylabel("shortcut feature")
handles, labels = axs[0].get_legend_handles_labels()
fg.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.1),
    ncols=2,
    markerscale=2,
    handletextpad=0.1,
    fontsize=7,
)
plotlib.save(fg, "d9_shift")

### Sweep: how strong is the shortcut? (plot `d9_accuracy`)

Train with a shortcut agreement $a$ between 0.5 (useless) and 0.99, test on the same distribution, on deployment data with agreement 0.5 and on data where the shortcut is reversed (agreement 0.05).

In [ ]:
rows = []
for a in (0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99):
    rng = np.random.default_rng(SEED)
    Xa, ya = make_data(2000, a, rng)
    m_both, m_core = LogisticRegression().fit(Xa, ya), LogisticRegression().fit(Xa[:, :1], ya)
    row = {"train_agreement": a, "w_core": m_both.coef_[0, 0], "w_shortcut": m_both.coef_[0, 1]}
    for name, test_a in [("same", a), ("shift_0.5", 0.5), ("reversed_0.05", 0.05)]:
        Xt, yt = make_data(4000, test_a, rng)
        row[f"both_{name}"], row[f"core_{name}"] = m_both.score(Xt, yt), m_core.score(Xt[:, :1], yt)
    rows.append(row)
sweep9 = pl.DataFrame(rows)
show(sweep9, 3)

In [ ]:
# plot d9_accuracy: left the two models before and after the shift, right the sweep of the training agreement
fg, axs = plotlib.fig("full", 1.9, ncols=2, gridspec_kw={"width_ratios": [0.9, 1.1]})
ax = axs[0]
xs, w = np.arange(2), 0.34
for j, (col, colour, lab) in enumerate(
    [("same_distribution", BLUE, "same distribution"), ("after_shift", RED, "after shift")]
):
    bars = ax.bar(xs + (j - 0.5) * w, acc9[col], w, color=colour, label=lab)
    for rect, v in zip(bars, acc9[col]):
        ax.annotate(
            f"{v:.2f}", (rect.get_x() + w / 2, v), (0, 2), textcoords="offset points", ha="center", fontsize=7.5
        )
ax.set(xticks=xs, xticklabels=["both\nfeatures", "core\nonly"], ylim=(0.4, 1.1), ylabel="accuracy")
ax.legend(
    loc="lower center",
    bbox_to_anchor=(0.5, 1.0),
    ncols=2,
    borderaxespad=0.1,
    columnspacing=0.8,
    handletextpad=0.3,
    fontsize=7,
)
ax.grid(axis="x", visible=False)
ax = axs[1]
a_ = sweep9["train_agreement"]
ax.plot(a_, sweep9["both_same"], color=BLUE, marker="o", ms=2.5, label="both, same distribution")
ax.plot(a_, sweep9["both_shift_0.5"], color=RED, marker="o", ms=2.5, label="both, shifted")
ax.plot(a_, sweep9["core_shift_0.5"], color=GREEN, marker="s", ms=2.5, label="core only, shifted")
ax.set(xlabel="shortcut agreement in training", ylabel="accuracy", ylim=(0.5, 1.02))
ax.legend(loc="lower left", borderaxespad=0.1, handletextpad=0.3, fontsize=7)
plotlib.save(fg, "d9_accuracy")

The model did what it was asked: minimize training error. The **shortcut** was the easiest pattern to use.
A validation set drawn from the same distribution **cannot detect** this failure; only data from the deployment conditions can.
The stronger the shortcut in training, the more the model leans on it and the more it loses after the shift; the model that never saw the shortcut loses nothing.

**Takeaways of D5-D9**

1. Too simple means high bias; too complex means high variance. Aim for the balance, and let more data move it.
2. No model is best everywhere (No Free Lunch).
3. Leakage produces great numbers from nothing; keep every learned step inside the validation loop.
4. Models learn correlations, not causes: predictions do not tell you what happens when you intervene.
5. When the distribution shifts, shortcuts break.